# Практика 4. Повторные испытания

Задачи 2.9–2.11: схема Бернулли, приближение Пуассона и формулы Муавра—Лапласа.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binom, norm, poisson

## Задача 2.9

В среднем 20% пакетов акций на аукционах продаются по первоначально заявленной цене. Для 9 пакетов найдём вероятности требуемых событий. Число проданных пакетов $X$ имеет биномиальное распределение: $X\sim Bin(9, 0{,}2)$.

In [ ]:
n, p = 9, 0.2

p_four_sold = binom.pmf(4, n, p)          # не проданы 5 пакетов
p_less_than_two = binom.cdf(1, n, p)      # X < 2
p_at_most_two = binom.cdf(2, n, p)        # X <= 2
p_at_least_two = binom.sf(1, n, p)        # X >= 2
modes = [1, 2]                            # (n + 1) * p = 2 — целое число

print(f"1) Не проданы 5 пакетов: {p_four_sold:.6f}")
print(f"2а) Продано менее 2:     {p_less_than_two:.6f}")
print(f"2б) Продано не более 2:  {p_at_most_two:.6f}")
print(f"2в) Продано хотя бы 2:   {p_at_least_two:.6f}")
for mode in modes:
    print(f"2г) P(X = {mode}) = {binom.pmf(mode, n, p):.6f}")

Если $(n+1)p$ — целое число, биномиальное распределение имеет две соседние моды: $(n+1)p-1$ и $(n+1)p$. Поэтому здесь наиболее вероятны одновременно 1 и 2 проданных пакета.

## Задача 2.10

Из 10 000 изделий в среднем повреждается 0,02%. Для редких событий используем приближение Пуассона с параметром

$$\lambda=np=10\,000\cdot0{,}0002=2.$$

In [ ]:
lam = 2

p_three_damaged = poisson.pmf(3, lam)
p_at_least_three_damaged = poisson.sf(2, lam)
p_exactly_9997_undamaged = poisson.pmf(3, lam)
p_at_least_9997_undamaged = poisson.cdf(3, lam)

print(f"1а) Повреждено 3:           {p_three_damaged:.6f}")
print(f"1б) Повреждено не менее 3:  {p_at_least_three_damaged:.6f}")
print(f"2а) Не повреждено 9997:     {p_exactly_9997_undamaged:.6f}")
print(f"2б) Не повреждено >= 9997:  {p_at_least_9997_undamaged:.6f}")

## Задача 2.11

Для 1000 предприятий вероятность нарушения равна $p=0{,}5$. Здесь $X\sim Bin(1000,0{,}5)$, математическое ожидание $np=500$, а стандартное отклонение $\sqrt{npq}=\sqrt{250}$.

Для отдельного значения применим локальную формулу Муавра—Лапласа, для диапазонов — интегральную формулу с поправкой на непрерывность.

In [ ]:
n, p = 1000, 0.5
mu = n * p
sigma = math.sqrt(n * p * (1 - p))

# Локальная формула Муавра—Лапласа
z_480 = (480 - mu) / sigma
p_480_approx = norm.pdf(z_480) / sigma
p_mode_approx = norm.pdf(0) / sigma

# Интегральная формула с поправкой на непрерывность
p_at_least_480_approx = norm.sf((479.5 - mu) / sigma)
p_480_to_520_approx = norm.cdf((520.5 - mu) / sigma) - norm.cdf((479.5 - mu) / sigma)

# Точные значения нужны для контроля качества приближения
p_480_exact = binom.pmf(480, n, p)
p_at_least_480_exact = binom.sf(479, n, p)
p_480_to_520_exact = binom.cdf(520, n, p) - binom.cdf(479, n, p)

print(f"а) P(X=480), приближённо: {p_480_approx:.6f}; точно: {p_480_exact:.6f}")
print(f"б) Мода X=500, P(X=500) ≈ {p_mode_approx:.6f}")
print(f"в) P(X>=480), приближённо: {p_at_least_480_approx:.6f}; точно: {p_at_least_480_exact:.6f}")
print(f"г) P(480<=X<=520), приближённо: {p_480_to_520_approx:.6f}; точно: {p_480_to_520_exact:.6f}")

In [ ]:
x = np.arange(450, 551)
exact_pmf = binom.pmf(x, n, p)
normal_density = norm.pdf((x - mu) / sigma) / sigma

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, exact_pmf, label="Точное биномиальное распределение", linewidth=2)
ax.plot(x, normal_density, "--", label="Нормальное приближение", linewidth=2)
ax.axvspan(480, 520, color="teal", alpha=0.12, label="Интервал [480, 520]")
ax.set(title="Муавр—Лаплас для Bin(1000, 0.5)", xlabel="Число предприятий с нарушениями", ylabel="Вероятность")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Вывод

Биномиальная модель применяется напрямую при небольшом числе испытаний, распределение Пуассона удобно для редких событий, а нормальное приближение хорошо описывает биномиальное распределение при большом $npq$. Поправка на непрерывность улучшает расчёт вероятностей на интервалах.